# 011 - Date & Time Functions

- Format
- Convert
- CAST

In [ ]:
import pandas as pd
import numpy as np

In [8]:
df_customers = pd.read_csv('data/sales_customers.csv')
df_employees = pd.read_csv('data/sales_employees.csv')
df_orders = pd.read_csv('data/sales_orders.csv')
df_orderarchive = pd.read_csv('data/sales_ordersarchive.csv')
df_products = pd.read_csv('data/sales_products.csv')

# FORMAT()

- Formats a date or time value

```SQL
SELECT
    orderid,
    creationtime,
    TO_CHAR(creationtime, 'MM-dd-yyyy') AS USA_Fomat,
    TO_CHAR(creationtime, 'dd-MM-yyyy') AS Euro_Format,
    TO_CHAR(creationtime, 'DD') AS dd,          -- Día del mes (01)
    TO_CHAR(creationtime, 'Dy') AS ddd,         -- Abrev. día (Wed)
    TO_CHAR(creationtime, 'Day') AS dddd,       -- Nombre día (Wednesday)
    TO_CHAR(creationtime, 'MM') AS mm,          -- Mes numérico (01)
    TO_CHAR(creationtime, 'Mon') AS mmm,        -- Abrev. mes (Jan) <--- ESTA ES LA CLAVE
    TO_CHAR(creationtime, 'Month') AS mmmm      -- Nombre mes (January)
FROM sales.orders;
```

In [4]:
# 1. Aseguramos que la columna sea datetime
df_orders['creationtime'] = pd.to_datetime(df_orders['creationtime'])

# 2. Creamos las columnas con los formatos solicitados
df_orders['USA_Format'] = df_orders['creationtime'].dt.strftime('%m-%d-%Y')
df_orders['Euro_Format'] = df_orders['creationtime'].dt.strftime('%d-%m-%Y')
df_orders['dd']         = df_orders['creationtime'].dt.strftime('%d')
df_orders['ddd']        = df_orders['creationtime'].dt.strftime('%a') # Abrev: Mon, Tue...
df_orders['dddd']       = df_orders['creationtime'].dt.day_name()     # Nombre completo: Monday...
df_orders['mm']         = df_orders['creationtime'].dt.strftime('%m')
df_orders['mmm']        = df_orders['creationtime'].dt.strftime('%b') # Abrev: Jan, Feb...
df_orders['mmmm']       = df_orders['creationtime'].dt.month_name()   # Nombre completo: January...

### SQL TASK

#### Show CreationTime using the following format: 

**Day Wed Jan Q1 2025 12:34.56 PM**

```SQL
SELECT
    orderid,
    creationtime,
    'Day ' || To_CHAR(creationtime, 'Dy Mon') || ' Q'|| DATE_PART('quarter', creationtime) || TO_CHAR(creationtime, ' yyyy hh:mm:ss') || ' PM'
FROM sales.orders;
```

In [7]:
# Aseguramos que sea datetime
df_orders['creationtime'] = pd.to_datetime(df_orders['creationtime'])

# Construimos la cadena compleja
df_orders['custom_label'] = (
    'Day ' + 
    df_orders['creationtime'].dt.strftime('%a %b') +      # 'Dy Mon' -> 'Fri Jan'
    ' Q' + 
    df_orders['creationtime'].dt.quarter.astype(str) +     # Quarter -> '1'
    df_orders['creationtime'].dt.strftime(' %Y %I:%M:%S %p') # ' yyyy hh:mm:ss PM'
)

df_orders

,orderid,productid,customerid,salespersonid,orderdate,shipdate,orderstatus,shipaddress,billaddress,quantity,...,creationtime,USA_Format,Euro_Format,dd,ddd,dddd,mm,mmm,mmmm,custom_label
0,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,...,2025-01-01 12:34:56,01-01-2025,01-01-2025,01,Wed,Wednesday,01,Jan,January,Day Wed Jan Q1 2025 12:34:56 PM
1,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,NaN,1,...,2025-01-05 23:22:04,01-05-2025,05-01-2025,05,Sun,Sunday,01,Jan,January,Day Sun Jan Q1 2025 11:22:04 PM
2,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,...,2025-01-10 18:24:08,01-10-2025,10-01-2025,10,Fri,Friday,01,Jan,January,Day Fri Jan Q1 2025 06:24:08 PM
3,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,NaN,2,...,2025-01-20 05:50:33,01-20-2025,20-01-2025,20,Mon,Monday,01,Jan,January,Day Mon Jan Q1 2025 05:50:33 AM
4,5,104,2,5,2025-02-01,2025-02-05,Delivered,NaN,NaN,1,...,2025-02-01 14:02:41,02-01-2025,01-02-2025,01,Sat,Saturday,02,Feb,February,Day Sat Feb Q1 2025 02:02:41 PM
5,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,NaN,2,...,2025-02-06 15:34:57,02-06-2025,06-02-2025,06,Thu,Thursday,02,Feb,February,Day Thu Feb Q1 2025 03:34:57 PM
6,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,NaN,2,...,2025-02-16 06:22:01,02-16-2025,16-02-2025,16,Sun,Sunday,02,Feb,February,Day Sun Feb Q1 2025 06:22:01 AM
7,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,...,2025-02-18 10:45:22,02-18-2025,18-02-2025,18,Tue,Tuesday,02,Feb,February,Day Tue Feb Q1 2025 10:45:22 AM
8,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,NaN,2,...,2025-03-10 12:59:04,03-10-2025,10-03-2025,10,Mon,Monday,03,Mar,March,Day Mon Mar Q1 2025 12:59:04 PM
9,10,102,3,5,2025-03-15,2025-03-20,Shipped,NaN,NaN,0,...,2025-03-16 23:25:15,03-16-2025,16-03-2025,16,Sun,Sunday,03,Mar,March,Day Sun Mar Q1 2025 11:25:15 PM


### CONVERT()

#### Converts a data or time value to a different data type & Formats the value

```SQL
SELECT
    '123'::INT AS string_to_int,
    '2025-08-20'::DATE AS string_to_date,
    creationtime,
    creationtime::DATE AS datetime_to_date,
    TO_CHAR(creationtime, 'MM-DD-YYYY') AS usa_std_stye,
    TO_CHAR(creationtime, 'DD-MM-YYYY') AS euro_std_style
FROM sales.orders
```

In [ ]:
df_orders['string_to_int'] = pd.Series(['123']).astype(int)
df_orders['string_to_date'] = pd.to_datetime('2025-08-20').date()

df_orders['creationtime'] = pd.to_datetime(df_orders['creationtime'])
df_orders['datetime_to_date'] = df_orders['creationtime'].dt.to_pydatetime

df_orders['usa_std_style'] = df_orders['creationtime'].dt.strftime('%m-%d-%Y')
df_orders['euro_std_style'] = df_orders['creationtime'].dt.strftime('%d-%m-%Y')


In [12]:
df_orders

,orderid,productid,customerid,salespersonid,orderdate,shipdate,orderstatus,shipaddress,billaddress,quantity,sales,creationtime,string_to_int,string_to_date,datetime_to_date,usa_std_style,euro_std_style
0,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10,2025-01-01 12:34:56,123.0,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,01-01-2025,01-01-2025
1,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,NaN,1,15,2025-01-05 23:22:04,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,01-05-2025,05-01-2025
2,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20,2025-01-10 18:24:08,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,01-10-2025,10-01-2025
3,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,NaN,2,60,2025-01-20 05:50:33,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,01-20-2025,20-01-2025
4,5,104,2,5,2025-02-01,2025-02-05,Delivered,NaN,NaN,1,25,2025-02-01 14:02:41,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,02-01-2025,01-02-2025
5,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,NaN,2,50,2025-02-06 15:34:57,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,02-06-2025,06-02-2025
6,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,NaN,2,30,2025-02-16 06:22:01,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,02-16-2025,16-02-2025
7,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90,2025-02-18 10:45:22,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,02-18-2025,18-02-2025
8,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,NaN,2,20,2025-03-10 12:59:04,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,03-10-2025,10-03-2025
9,10,102,3,5,2025-03-15,2025-03-20,Shipped,NaN,NaN,0,60,2025-03-16 23:25:15,NaN,2025-08-20,<bound method DatetimeProperties.to_pydatetime...,03-16-2025,16-03-2025


### CAST

#### Converts a value to a specified data type

```SQL
SELECT
    CAST('123' AS INT) AS string_to_int,
    CAST(123 AS VARCHAR) AS int_to_sting,
    CAST('2025-08-20' AS DATE) AS sting_to_date,
    CAST('2025-08-20' AS TIMESTAMP) AS sting_to_datetime,
    creationtime,
    CAST(creationtime AS DATE) AS datetime_to_date
FROM sales.orders
```

In [13]:
# 1. String a Int
df_orders['string_to_int'] = pd.Series(['123']).astype(int)

# 2. Int a String (Varchar)
df_orders['int_to_string'] = pd.Series([123]).astype(str)

# 3. String a Date
# Usamos pd.to_datetime y luego extraemos solo la fecha (.dt.date)
df_orders['string_to_date'] = pd.to_datetime('2025-08-20').date()

# 4. String a Datetime (Equivalente a TIMESTAMP)
df_orders['string_to_datetime'] = pd.to_datetime('2025-08-20')

# 5. Columna Datetime a Date
# Asumiendo que creationtime ya es datetime, extraemos la parte de la fecha
df_orders['creationtime'] = pd.to_datetime(df_orders['creationtime'])
df_orders['datetime_to_date'] = df_orders['creationtime'].dt.date

In [14]:
df_orders

,orderid,productid,customerid,salespersonid,orderdate,shipdate,orderstatus,shipaddress,billaddress,quantity,sales,creationtime,string_to_int,string_to_date,datetime_to_date,usa_std_style,euro_std_style,int_to_string,string_to_datetime
0,1,101,2,3,2025-01-01,2025-01-05,Delivered,9833 Mt. Dias Blv.,1226 Shoe St.,1,10,2025-01-01 12:34:56,123.0,2025-08-20,2025-01-01,01-01-2025,01-01-2025,123,2025-08-20
1,2,102,3,3,2025-01-05,2025-01-10,Shipped,250 Race Court,NaN,1,15,2025-01-05 23:22:04,NaN,2025-08-20,2025-01-05,01-05-2025,05-01-2025,NaN,2025-08-20
2,3,101,1,5,2025-01-10,2025-01-25,Delivered,8157 W. Book,8157 W. Book,2,20,2025-01-10 18:24:08,NaN,2025-08-20,2025-01-10,01-10-2025,10-01-2025,NaN,2025-08-20
3,4,105,1,3,2025-01-20,2025-01-25,Shipped,5724 Victory Lane,NaN,2,60,2025-01-20 05:50:33,NaN,2025-08-20,2025-01-20,01-20-2025,20-01-2025,NaN,2025-08-20
4,5,104,2,5,2025-02-01,2025-02-05,Delivered,NaN,NaN,1,25,2025-02-01 14:02:41,NaN,2025-08-20,2025-02-01,02-01-2025,01-02-2025,NaN,2025-08-20
5,6,104,3,5,2025-02-05,2025-02-10,Delivered,1792 Belmont Rd.,NaN,2,50,2025-02-06 15:34:57,NaN,2025-08-20,2025-02-06,02-06-2025,06-02-2025,NaN,2025-08-20
6,7,102,1,1,2025-02-15,2025-02-27,Delivered,136 Balboa Court,NaN,2,30,2025-02-16 06:22:01,NaN,2025-08-20,2025-02-16,02-16-2025,16-02-2025,NaN,2025-08-20
7,8,101,4,3,2025-02-18,2025-02-27,Shipped,2947 Vine Lane,4311 Clay Rd,3,90,2025-02-18 10:45:22,NaN,2025-08-20,2025-02-18,02-18-2025,18-02-2025,NaN,2025-08-20
8,9,101,2,3,2025-03-10,2025-03-15,Shipped,3768 Door Way,NaN,2,20,2025-03-10 12:59:04,NaN,2025-08-20,2025-03-10,03-10-2025,10-03-2025,NaN,2025-08-20
9,10,102,3,5,2025-03-15,2025-03-20,Shipped,NaN,NaN,0,60,2025-03-16 23:25:15,NaN,2025-08-20,2025-03-16,03-16-2025,16-03-2025,NaN,2025-08-20
